# Exploratory data analysis of the ICU cohorts

This notebook describes every database used in Pipeline B, at two levels:

1. **YAIB-cohorts output** (`$IAN_DATA/cohorts/<task>/<src>/{dyn,sta,outc}.parquet`) for `mortality24` and `aki`.
2. **Raw stay metadata** (`$IAN_DATA/raw/<src>/...`), read with the same code the pipeline uses
   (`icu_aes_nas.data.sources.load_stay_metadata`) plus a few cheap extra columns (length of stay, unit type).

Sections: inventory, cohort size and attrition, label prevalence, statics, missingness (per variable and over
time), variable distributions, MIMIC-IV vs external database shift, length of stay, summary table.

**Runs on CPU in under 8 GB.** Dynamic tables are read one column at a time with pyarrow, so the full
`dyn.parquet` is never held in memory.

**Databases.** MIMIC-IV and eICU are on. HiRID and AUMCdb are commented out in the config cell below until
access is granted and their cohorts are extracted. Uncommenting them is the only change needed.

**Headless run** (from the repo root, inside `.venv`):

```bash
jupyter nbconvert --to notebook --execute notebooks/eda.ipynb \
    --output-dir outputs/eda --output eda_executed.ipynb --ExecutePreprocessor.timeout=-1
# or on the cluster:  sbatch slurm/eda.sbatch
```

Figures go to `outputs/eda/figures/` (PDF for the thesis, PNG for quick viewing) and tables to
`outputs/eda/tables/` (CSV, plus `.tex` for the summary table). Both stay inside `outputs/`, which git ignores.
Only aggregate statistics are written. Check your data use agreement before copying any of it off the cluster
(PhysioNet forbids publishing cells with fewer than 10 patients; the notebook never prints individual rows).

## 0. Configuration

In [ ]:
import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)


def _find_root() -> Path:
    if os.environ.get("IAN_ROOT"):
        return Path(os.environ["IAN_ROOT"]).resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "icu_aes_nas").exists():
            return p
    return here


ROOT = _find_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from icu_aes_nas.constants import DYNAMIC_VARS, NUMERIC_STATICS  # noqa: E402
from icu_aes_nas.data.sources import load_stay_metadata  # noqa: E402

DATA = Path(os.environ.get("IAN_DATA", ROOT / "data"))
COHORTS = DATA / "cohorts"
RAW = DATA / "raw"
OUT = Path(os.environ.get("EDA_OUT", ROOT / "outputs" / "eda"))
FIG = OUT / "figures"
TAB = OUT / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

SOURCES = [
    "miiv",
    "eicu",
    "hirid",     # access granted 2026-11-18; skipped with a note until `extract_cohorts.sbatch hirid` has run
    # "aumc",    # uncomment once AUMCdb access is granted and `extract_cohorts.sbatch aumc` has run
]
TASKS = ["mortality24", "aki"]
# Optional overrides for headless runs, e.g. EDA_SOURCES=miiv,eicu,hirid EDA_TASKS=mortality24
if os.environ.get("EDA_SOURCES"):
    SOURCES = [s.strip() for s in os.environ["EDA_SOURCES"].split(",") if s.strip()]
if os.environ.get("EDA_TASKS"):
    TASKS = [t.strip() for t in os.environ["EDA_TASKS"].split(",") if t.strip()]

FIRST_STAY_ONLY = True          # same rule as data.first_stay_only in conf/data/pipeline_b.yaml
REF_SOURCE = "miiv"             # training database in the shift comparison
SHIFT_TASK = "mortality24"      # cohort used for the shift analysis and the domain classifier
MAX_STAYS_CLASSIFIER = 20_000   # per source, for the domain classifier
SAMPLE_PER_VAR = 200_000        # values kept per (task, source, variable) for histograms and KS tests
TEMPORAL_MIN_YEAR = 2017        # data.temporal_holdout.min_year
SEED = 0
N_THREADS = int(os.environ.get("OMP_NUM_THREADS", "4"))
rng = np.random.default_rng(SEED)

NAMES = {"miiv": "MIMIC-IV", "eicu": "eICU-CRD", "hirid": "HiRID", "aumc": "AUMCdb"}
# Fixed colour per database (categorical slots 1-4), so a database keeps its colour in every figure.
COLORS = {"miiv": "#2a78d6", "eicu": "#eb6834", "hirid": "#1baf7a", "aumc": "#eda100"}
TASK_NAMES = {"mortality24": "Mortality (24 h)", "aki": "AKI"}

# Drop sources/tasks whose cohort folder does not exist (yet), with a note instead of a crash.
AVAILABLE = {}
for task in TASKS:
    AVAILABLE[task] = []
    for src in SOURCES:
        p = COHORTS / task / src
        if all((p / f).exists() for f in ("dyn.parquet", "sta.parquet", "outc.parquet")):
            AVAILABLE[task].append(src)
        else:
            print(f"[skip] {task}/{src}: no cohort files in {p}")
TASKS = [t for t in TASKS if AVAILABLE[t]]
SOURCES = [s for s in SOURCES if any(s in AVAILABLE[t] for t in TASKS)]
if SHIFT_TASK not in TASKS and TASKS:
    SHIFT_TASK = TASKS[0]
print("ROOT   ", ROOT)
print("DATA   ", DATA)
print("OUT    ", OUT)
print("tasks  ", TASKS)
print("sources", SOURCES, "| available per task:", AVAILABLE)
assert TASKS, "No cohort found. Run the YAIB extraction (README section 5) or `ian-make-synthetic` first."
T0 = time.time()

In [ ]:
# Plot style: thesis text width is about 6.3 in.
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 1.6,
    "pdf.fonttype": 42,
})
WIDTH = 6.3


def save(fig, name: str):
    fig.savefig(FIG / f"{name}.pdf")
    fig.savefig(FIG / f"{name}.png")
    plt.show()
    plt.close(fig)


def save_table(df: pd.DataFrame, name: str, index=True):
    df.to_csv(TAB / f"{name}.csv", index=index)
    return df


def legend_sources(ax, sources, **kw):
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=COLORS[s], lw=2) for s in sources]
    ax.legend(handles, [NAMES[s] for s in sources], frameon=False, **kw)


def fig_legend(fig, sources):
    """One legend under a multi-panel figure, so it never covers data."""
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=COLORS[s], lw=2) for s in sources]
    fig.legend(handles, [NAMES[s] for s in sources], loc="upper center", bbox_to_anchor=(0.5, 0.0),
               ncol=len(sources), frameon=False)

### Readers

`time` comes from ricu as a difftime in hours and can arrive as a duration, so it is converted the same way as in
`icu_aes_nas.data.cohort`. Values are cast to float32 to halve memory.

In [ ]:
def cohort_file(task, src, kind):
    return COHORTS / task / src / f"{kind}.parquet"


def parquet_columns(path):
    return pq.ParquetFile(path).schema_arrow.names


def to_hours(col: pd.Series) -> np.ndarray:
    if pd.api.types.is_timedelta64_dtype(col):
        h = col.dt.total_seconds() / 3600
    else:
        h = pd.to_numeric(col, errors="coerce")
    return np.round(h.to_numpy(dtype="float64")).astype("int32")


def read_cols(path, cols) -> pd.DataFrame:
    cols = [c for c in cols if c in parquet_columns(path)]
    df = pq.read_table(path, columns=cols).to_pandas(self_destruct=True)
    if "time" in df:
        df["time"] = to_hours(df["time"])
    for c in df.columns:
        if df[c].dtype == "float64":
            df[c] = df[c].astype("float32")
    return df


def read_one(path, col) -> np.ndarray:
    arr = pq.read_table(path, columns=[col]).column(0).to_numpy(zero_copy_only=False)
    return pd.to_numeric(pd.Series(arr), errors="coerce").to_numpy(dtype="float32")


def same_kind(ids, like: pd.Series) -> np.ndarray:
    """Cast metadata stay ids (str) to the dtype of the cohort's stay_id column, so isin() matches."""
    ids = pd.Series(list(ids), dtype=str)
    if pd.api.types.is_numeric_dtype(like):
        return pd.to_numeric(ids, errors="coerce").dropna().astype("int64").to_numpy()
    return ids.to_numpy()

## 1. Inventory

File sizes and row counts from the parquet footers (no data is loaded).

In [ ]:
rows = []
for task in TASKS:
    for src in AVAILABLE[task]:
        for kind in ("dyn", "sta", "outc"):
            f = cohort_file(task, src, kind)
            meta = pq.ParquetFile(f).metadata
            rows.append({"task": task, "source": src, "file": kind, "rows": meta.num_rows,
                         "columns": meta.num_columns, "MB": round(f.stat().st_size / 2**20, 1)})
inventory = save_table(pd.DataFrame(rows), "inventory", index=False)
display(inventory)

for task in TASKS:
    for src in AVAILABLE[task]:
        cols = parquet_columns(cohort_file(task, src, "dyn"))
        missing = [v for v in DYNAMIC_VARS if v not in cols]
        extra = [c for c in cols if c not in DYNAMIC_VARS and c not in ("stay_id", "time")]
        print(f"{task}/{src}: dyn has {len(cols)} columns; missing concepts: {missing or 'none'}; "
              f"extra: {extra or 'none'}")
        print(f"   sta: {parquet_columns(cohort_file(task, src, 'sta'))}   "
              f"outc: {parquet_columns(cohort_file(task, src, 'outc'))}")

## 2. Cohort size and attrition

Raw ICU stays and patients come from the PhysioNet tables in `raw/<src>/`. The YAIB cohort applies the task's
inclusion criteria (adults, minimum length of stay, enough data). The pipeline then keeps only each patient's
**first ICU stay**, decided over all of the patient's stays, not only those in the cohort (same rule as
`icu_aes_nas.data.cohort.build_source_cohort`).

In [ ]:
META = {}
for src in SOURCES:
    try:
        META[src] = load_stay_metadata(src, RAW)
        print(f"{src}: {len(META[src]):,} raw stays, {META[src]['patient_id'].nunique():,} patients")
    except Exception as e:  # raw tables not downloaded on this machine
        print(f"[no raw metadata] {src}: {type(e).__name__}: {e}")

FIRST_STAYS = {}
for src, m in META.items():
    FIRST_STAYS[src] = set(m.sort_values("stay_order").groupby("patient_id", sort=False).head(1)["stay_id"])

KEEP = {}         # (task, src) -> stay ids (in the cohort's dtype) used in every later section
attr_rows = []
for task in TASKS:
    for src in AVAILABLE[task]:
        sta_ids = read_cols(cohort_file(task, src, "sta"), ["stay_id"])["stay_id"].drop_duplicates()
        row = {"task": task, "source": src,
               "raw_stays": len(META[src]) if src in META else np.nan,
               "raw_patients": META[src]["patient_id"].nunique() if src in META else np.nan,
               "yaib_cohort": len(sta_ids)}
        if FIRST_STAY_ONLY and src in META:
            keep = set(sta_ids.astype(str)) & FIRST_STAYS[src]
            row["not_first_icu_stay"] = row["yaib_cohort"] - len(keep)
            KEEP[(task, src)] = same_kind(keep, sta_ids)
        else:
            if FIRST_STAY_ONLY:
                print(f"[warn] {task}/{src}: no raw metadata, so the first-stay filter cannot be applied")
            row["not_first_icu_stay"] = np.nan
            KEEP[(task, src)] = sta_ids.to_numpy()
        row["final"] = len(KEEP[(task, src)])
        if src in META:
            m = META[src][META[src]["stay_id"].isin(set(pd.Series(KEEP[(task, src)]).astype(str)))]
            row["final_patients"] = m["patient_id"].nunique()
            row["hospitals"] = m["hospital_id"].nunique()
        attr_rows.append(row)
attrition = save_table(pd.DataFrame(attr_rows), "attrition", index=False)
display(attrition)

In [ ]:
stages = [("raw_stays", "Raw ICU stays"), ("yaib_cohort", "YAIB cohort"), ("final", "First ICU stay")]
fig, axes = plt.subplots(1, len(TASKS), figsize=(WIDTH, 2.6), squeeze=False, sharey=False)
for ax, task in zip(axes[0], TASKS):
    a = attrition[attrition.task == task].set_index("source")
    srcs = list(a.index)
    w = 0.8 / len(srcs)
    x = np.arange(len(stages))
    for i, src in enumerate(srcs):
        vals = [a.loc[src, k] for k, _ in stages]
        ax.bar(x + (i - (len(srcs) - 1) / 2) * w, vals, width=w * 0.92, color=COLORS[src], label=NAMES[src])
    ax.set_xticks(x, [lbl for _, lbl in stages])
    ax.set_title(TASK_NAMES.get(task, task))
    ax.set_ylabel("Stays")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}k" if v >= 1000 else f"{v:.0f}"))
fig.tight_layout()
from matplotlib.patches import Patch
fig.legend([Patch(color=COLORS[s]) for s in SOURCES], [NAMES[s] for s in SOURCES], loc="upper center",
           bbox_to_anchor=(0.5, 0.0), ncol=len(SOURCES), frameon=False)
save(fig, "attrition")

### Stays per patient, estimated admission year, hospitals

The estimated year drives the MIMIC-IV temporal holdout (`est_year >= 2017`, about +-1.5 y accuracy). eICU spans
2014-2015 and many hospitals, which is what the hospital-level bootstrap resamples.

In [ ]:
rows = []
for src, m in META.items():
    spp = m.groupby("patient_id").size()
    yrs = m["est_year"].dropna()
    rows.append({"source": src, "stays": len(m), "patients": m["patient_id"].nunique(),
                 "stays_per_patient_mean": spp.mean(), "patients_with_>1_stay_%": 100 * (spp > 1).mean(),
                 "hospitals": m["hospital_id"].nunique(),
                 "est_year_min": yrs.min() if len(yrs) else np.nan,
                 "est_year_max": yrs.max() if len(yrs) else np.nan,
                 f"est_year_>={TEMPORAL_MIN_YEAR}_%": 100 * (yrs >= TEMPORAL_MIN_YEAR).mean() if len(yrs) else np.nan})
if rows:
    display(save_table(pd.DataFrame(rows), "raw_metadata_overview", index=False).round(2))

with_year = [s for s, m in META.items() if m["est_year"].notna().any()]
with_hosp = [s for s, m in META.items() if m["hospital_id"].nunique() > 1]
n_pan = len(with_year) + len(with_hosp)
if n_pan:
    fig, axes = plt.subplots(1, n_pan, figsize=(WIDTH, 2.4), squeeze=False)
    k = 0
    for src in with_year:
        ax = axes[0][k]; k += 1
        y = META[src]["est_year"].dropna().round().astype(int).value_counts().sort_index()
        ax.bar(y.index, y.values, color=COLORS[src], width=0.85)
        if src == "miiv":
            ax.axvline(TEMPORAL_MIN_YEAR - 0.5, color="#555", lw=1, ls="--")
            ax.text(TEMPORAL_MIN_YEAR - 0.4, ax.get_ylim()[1] * 0.95, "temporal\nholdout", va="top", fontsize=7)
        ax.set_title(f"{NAMES[src]}: estimated year of stay")
        ax.set_ylabel("Stays")
    for src in with_hosp:
        ax = axes[0][k]; k += 1
        per_h = META[src].groupby("hospital_id").size().sort_values(ascending=False).to_numpy()
        ax.bar(np.arange(len(per_h)), per_h, color=COLORS[src], width=1.0)
        ax.set_title(f"{NAMES[src]}: stays per hospital ({len(per_h)} hospitals)")
        ax.set_xlabel("Hospital (sorted)")
        ax.set_ylabel("Stays")
    fig.tight_layout()
    save(fig, "raw_year_and_hospitals")

## 3. Labels

`mortality24`: one label per stay. `aki`: one label per hour; a stay counts as positive if any hour is positive.
For AKI we also show when the first positive hour occurs and the positive rate per hour since admission.

In [ ]:
LABELS = {}       # (task, src) -> stay-level label Series
AKI_ONSET = {}    # src -> first positive hour per positive stay
AKI_HOURLY = {}   # src -> positive rate per hour
prev_rows = []
for task in TASKS:
    for src in AVAILABLE[task]:
        outc = read_cols(cohort_file(task, src, "outc"), ["stay_id", "time", "label"])
        outc = outc[outc["stay_id"].isin(KEEP[(task, src)])]
        stay_lab = outc.groupby("stay_id")["label"].max().astype(int)
        LABELS[(task, src)] = stay_lab
        row = {"task": task, "source": src, "stays": len(stay_lab), "positive_stays": int(stay_lab.sum()),
               "stay_prevalence_%": 100 * stay_lab.mean()}
        if "time" in outc and outc.groupby("stay_id").size().max() > 1:   # dynamic task
            row["label_rows"] = len(outc)
            row["hour_prevalence_%"] = 100 * outc["label"].mean()
            pos = outc[outc["label"] == 1]
            AKI_ONSET[(task, src)] = pos.groupby("stay_id")["time"].min()
            row["median_onset_hour"] = AKI_ONSET[(task, src)].median()
            AKI_HOURLY[(task, src)] = outc[outc["time"] >= 0].groupby("time")["label"].agg(["mean", "size"])
        prev_rows.append(row)
        del outc
gc.collect()
prevalence = save_table(pd.DataFrame(prev_rows), "label_prevalence", index=False)
display(prevalence.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(WIDTH * 0.5, 2.4))
w = 0.8 / max(1, len(SOURCES))
for i, src in enumerate(SOURCES):
    p = prevalence[prevalence.source == src].set_index("task").reindex(TASKS)["stay_prevalence_%"]
    xs = np.arange(len(TASKS)) + (i - (len(SOURCES) - 1) / 2) * w
    bars = ax.bar(xs, p.values, width=w * 0.92, color=COLORS[src], label=NAMES[src])
    ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=1)
ax.set_xticks(np.arange(len(TASKS)), [TASK_NAMES.get(t, t) for t in TASKS])
ax.set_ylabel("Positive stays (%)")
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "label_prevalence")

dyn_tasks = sorted({t for t, _ in AKI_HOURLY})
for task in dyn_tasks:
    fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.4))
    srcs = [s for s in SOURCES if (task, s) in AKI_HOURLY]
    for src in srcs:
        on = AKI_ONSET[(task, src)]
        on = on[on >= 0]
        if len(on):
            axes[0].hist(on, bins=np.arange(0, on.max() + 7, 6), density=True, histtype="step",
                         lw=1.6, color=COLORS[src])
        h = AKI_HOURLY[(task, src)]
        h = h[h["size"] >= 50]                      # hide hours with few stays left
        axes[1].plot(h.index, 100 * h["mean"], color=COLORS[src])
    axes[0].set_xlabel("Hour of first positive label"); axes[0].set_ylabel("Density")
    axes[0].set_title("Onset among positive stays")
    axes[1].set_xlabel("Hours since ICU admission"); axes[1].set_ylabel("Positive hours (%)")
    axes[1].set_title("Hourly positive rate (hours with >= 50 stays)")
    legend_sources(axes[1], srcs)
    fig.tight_layout()
    save(fig, f"{task}_onset_and_hourly_rate")

## 4. Static variables

Age, sex, height and weight as harmonised by ricu. The table gives median [IQR], missingness and the number of
values outside a plausible adult range (age 18-100 y, height 120-230 cm, weight 30-300 kg), which hints at unit or
entry errors.

In [ ]:
PLAUSIBLE = {"age": (18, 100), "height": (120, 230), "weight": (30, 300)}
STA = {}
rows = []
for task in TASKS:
    for src in AVAILABLE[task]:
        sta = read_cols(cohort_file(task, src, "sta"), ["stay_id", "age", "sex", "height", "weight"])
        sta = sta[sta["stay_id"].isin(KEEP[(task, src)])].drop_duplicates("stay_id")
        STA[(task, src)] = sta
        r = {"task": task, "source": src, "stays": len(sta)}
        for v in NUMERIC_STATICS:
            if v not in sta:
                r[f"{v}_median[IQR]"] = "n/a"; continue
            x = pd.to_numeric(sta[v], errors="coerce")
            q = x.quantile([0.25, 0.5, 0.75])
            r[f"{v}_median[IQR]"] = f"{q[0.5]:.0f} [{q[0.25]:.0f}-{q[0.75]:.0f}]"
            r[f"{v}_missing_%"] = round(100 * x.isna().mean(), 1)
            lo, hi = PLAUSIBLE[v]
            r[f"{v}_implausible_n"] = int(((x < lo) | (x > hi)).sum())
        if "sex" in sta:
            s = sta["sex"].astype(str).str.lower()
            r["male_%"] = round(100 * s.isin(["male", "m", "1"]).mean(), 1)
            r["sex_missing_%"] = round(100 * sta["sex"].isna().mean(), 1)
        rows.append(r)
statics = save_table(pd.DataFrame(rows), "statics", index=False)
display(statics)

In [ ]:
for task in TASKS:
    srcs = AVAILABLE[task]
    fig, axes = plt.subplots(1, 4, figsize=(WIDTH, 2.2), gridspec_kw={"width_ratios": [1, 1, 1, 0.6]})
    for ax, v in zip(axes[:3], NUMERIC_STATICS):
        pooled = pd.concat([pd.to_numeric(STA[(task, s)][v], errors="coerce") for s in srcs if v in STA[(task, s)]])
        lo, hi = pooled.quantile([0.005, 0.995]) if pooled.notna().any() else (0, 1)
        bins = np.linspace(lo, hi, 40)
        for s in srcs:
            if v in STA[(task, s)]:
                x = pd.to_numeric(STA[(task, s)][v], errors="coerce").dropna()
                ax.hist(x.clip(lo, hi), bins=bins, density=True, histtype="step", lw=1.6, color=COLORS[s])
        ax.set_title({"age": "Age (years)", "height": "Height (cm)", "weight": "Weight (kg)"}[v])
        ax.set_yticks([])
    ax = axes[3]
    male = [statics[(statics.task == task) & (statics.source == s)]["male_%"].iloc[0] for s in srcs]
    bars = ax.bar(range(len(srcs)), male, color=[COLORS[s] for s in srcs], width=0.7)
    ax.bar_label(bars, fmt="%.0f", fontsize=7, padding=1)
    ax.set_xticks(range(len(srcs)), [NAMES[s] for s in srcs], rotation=30, ha="right")
    ax.set_title("Male (%)"); ax.set_ylim(0, 100)
    fig.suptitle(f"Statics: {TASK_NAMES.get(task, task)} cohort", y=1.02)
    fig.tight_layout()
    fig_legend(fig, srcs)
    save(fig, f"{task}_statics")

## 5. Dynamic variables: one pass over `dyn.parquet`

For each (task, database) the `stay_id`/`time` columns are read once and then each of the 48 variables one at a
time. From each variable we keep: missingness per row and per stay, missingness per hour since admission, exact
summary statistics, a random sample of values (for histograms and KS tests) and, for the shift cohort, the
per-stay mean and number of measurements (features of the domain classifier in section 7).

"Rows" are the stay-hours present in `dyn.parquet` (YAIB writes an hourly grid), so "% of rows observed" is the
share of hours with at least one measurement.

In [ ]:
QS = [0.005, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995]
MISS, OVERTIME, STATS, SAMPLES, HOURS, STAY_AGG = {}, {}, {}, {}, {}, {}

for task in TASKS:
    for src in AVAILABLE[task]:
        t_start = time.time()
        path = cohort_file(task, src, "dyn")
        base = read_cols(path, ["stay_id", "time"])
        mask = base["stay_id"].isin(KEEP[(task, src)]).to_numpy()
        base = base[mask].reset_index(drop=True)
        codes, uniq = pd.factorize(base["stay_id"])
        n_stays = len(uniq)
        hours = base["time"].to_numpy()
        HOURS[(task, src)] = pd.Series(hours).groupby(codes).max() + 1          # observed hours per stay
        t_ok = hours >= 0
        t_idx = np.where(t_ok, hours, 0)
        rows_per_t = np.bincount(t_idx[t_ok], minlength=1)
        cols_avail = set(parquet_columns(path))
        miss, ot, stats, samples = {}, {}, {}, {}
        agg_mean, agg_cnt = {}, {}
        for v in DYNAMIC_VARS:
            if v not in cols_avail:
                continue
            x = read_one(path, v)[mask]
            obs = ~np.isnan(x)
            cnt_per_stay = np.bincount(codes, weights=obs, minlength=n_stays)
            miss[v] = {"rows_observed_%": 100 * obs.mean(),
                       "stays_with_any_%": 100 * (cnt_per_stay > 0).mean(),
                       "median_obs_per_stay": float(np.median(cnt_per_stay[cnt_per_stay > 0]))
                       if (cnt_per_stay > 0).any() else 0.0}
            ot[v] = np.bincount(t_idx[t_ok], weights=obs[t_ok], minlength=len(rows_per_t)) / np.maximum(rows_per_t, 1)
            xv = x[obs]
            if len(xv):
                qs = np.quantile(xv, QS)
                stats[v] = {"n": len(xv), "mean": float(xv.mean(dtype="float64")),
                            "std": float(xv.std(dtype="float64")), "min": float(xv.min()), "max": float(xv.max()),
                            **{f"p{100*q:g}": float(val) for q, val in zip(QS, qs)}}
                samples[v] = xv if len(xv) <= SAMPLE_PER_VAR else rng.choice(xv, SAMPLE_PER_VAR, replace=False)
            if task == SHIFT_TASK:
                s = np.bincount(codes, weights=np.where(obs, x, 0), minlength=n_stays)
                with np.errstate(invalid="ignore", divide="ignore"):
                    agg_mean[v] = np.where(cnt_per_stay > 0, s / cnt_per_stay, np.nan).astype("float32")
                agg_cnt[v] = cnt_per_stay.astype("float32")
            del x, obs, xv
        MISS[(task, src)] = pd.DataFrame(miss).T
        OVERTIME[(task, src)] = pd.DataFrame(ot, index=np.arange(len(rows_per_t))).assign(n_rows=rows_per_t)
        STATS[(task, src)] = pd.DataFrame(stats).T
        SAMPLES[(task, src)] = samples
        if task == SHIFT_TASK:
            STAY_AGG[src] = pd.concat([pd.DataFrame(agg_mean, index=uniq).add_prefix("mean_"),
                                       pd.DataFrame(agg_cnt, index=uniq).add_prefix("n_")], axis=1)
        del base, codes, hours, t_ok, t_idx
        gc.collect()
        print(f"{task}/{src}: {mask.sum():,} rows, {n_stays:,} stays, {len(miss)} variables "
              f"in {time.time() - t_start:.0f}s")

### 5a. Missingness per variable

In [ ]:
for task in TASKS:
    srcs = AVAILABLE[task]
    tab = pd.concat({NAMES[s]: MISS[(task, s)] for s in srcs}, axis=1)
    save_table(tab.round(2), f"{task}_missingness")
    display(tab.round(1))

    order = MISS[(task, srcs[0])]["rows_observed_%"].sort_values(ascending=False).index
    fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 8.2), sharey=True)
    for ax, metric, title in zip(axes, ["rows_observed_%", "stays_with_any_%"],
                                 ["% of hours with a value", "% of stays with >= 1 value"]):
        mat = np.column_stack([MISS[(task, s)][metric].reindex(order).to_numpy() for s in srcs])
        im = ax.imshow(mat, aspect="auto", cmap="Blues", vmin=0, vmax=100)
        for (i, j), val in np.ndenumerate(mat):
            if not np.isnan(val):
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6,
                        color="white" if val > 60 else "#0b0b0b")
        ax.set_xticks(range(len(srcs)), [NAMES[s] for s in srcs])
        ax.set_yticks(range(len(order)), order)
        ax.set_title(title); ax.grid(False)
    fig.colorbar(im, ax=axes, shrink=0.4, label="%")
    fig.suptitle(f"Measurement coverage: {TASK_NAMES.get(task, task)} cohort", y=0.995)
    save(fig, f"{task}_missingness_heatmap")

### 5b. Missingness over time since ICU admission

Share of stays (still in the data at that hour) with a value, for vitals and common labs.

In [ ]:
TIME_VARS = ["hr", "map", "resp", "o2sat", "temp", "urine", "crea", "bun", "lact", "glu", "k", "wbc"]
for task in TASKS:
    srcs = AVAILABLE[task]
    vs = [v for v in TIME_VARS if all(v in OVERTIME[(task, s)] for s in srcs)]
    ncol = 4; nrow = int(np.ceil(len(vs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(WIDTH, 1.55 * nrow + 0.4), sharex=True, sharey=True, squeeze=False)
    for ax, v in zip(axes.flat, vs):
        for s in srcs:
            o = OVERTIME[(task, s)]
            o = o[o["n_rows"] >= 50]
            ax.plot(o.index, 100 * o[v], color=COLORS[s], lw=1.3)
        ax.set_title(v); ax.set_ylim(0, 100)
    for ax in axes.flat[len(vs):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("Hour")
    for ax in axes[:, 0]:
        ax.set_ylabel("% observed")
    fig.suptitle(f"Coverage over time: {TASK_NAMES.get(task, task)} cohort", y=1.0)
    fig.tight_layout()
    fig_legend(fig, srcs)
    save(fig, f"{task}_missingness_over_time")

## 6. Variable distributions

Exact quantiles come from all values; histograms use a random sample of up to `SAMPLE_PER_VAR` values per
variable, clipped to the pooled 0.5-99.5 percentile range so outliers do not flatten the plot. Units are ricu's
harmonised units, so differences between databases are real differences (or remaining unit problems; look at
`min`/`max` in the table).

In [ ]:
for task in TASKS:
    srcs = AVAILABLE[task]
    tab = pd.concat({NAMES[s]: STATS[(task, s)] for s in srcs}, names=["database", "variable"])
    tab = tab.reorder_levels(["variable", "database"])
    rank = {v: i for i, v in enumerate(DYNAMIC_VARS)}
    tab = tab.iloc[sorted(range(len(tab)), key=lambda i: (rank[tab.index[i][0]], tab.index[i][1]))]
    save_table(tab.round(3), f"{task}_variable_stats")
    display(tab[["n", "mean", "std", "p1", "p25", "p50", "p75", "p99", "min", "max"]].round(2).head(30))

    vs = [v for v in DYNAMIC_VARS if any(v in SAMPLES[(task, s)] for s in srcs)]
    ncol = 6; nrow = int(np.ceil(len(vs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(WIDTH, 1.05 * nrow + 0.4), squeeze=False)
    for ax, v in zip(axes.flat, vs):
        pooled = np.concatenate([SAMPLES[(task, s)][v] for s in srcs if v in SAMPLES[(task, s)]])
        lo, hi = np.quantile(pooled, [0.005, 0.995])
        if hi <= lo:
            hi = lo + 1
        bins = np.linspace(lo, hi, 35)
        for s in srcs:
            if v in SAMPLES[(task, s)]:
                ax.hist(np.clip(SAMPLES[(task, s)][v], lo, hi), bins=bins, density=True, histtype="step",
                        lw=1.1, color=COLORS[s])
        ax.set_title(v, pad=2); ax.set_yticks([]); ax.tick_params(axis="x", labelsize=6, pad=1)
        ax.grid(False)
    for ax in axes.flat[len(vs):]:
        ax.set_visible(False)
    fig.suptitle(f"Distributions of the 48 dynamic variables: {TASK_NAMES.get(task, task)} cohort", y=1.0)
    fig.tight_layout(h_pad=0.6, w_pad=0.4)
    fig_legend(fig, srcs)
    save(fig, f"{task}_variable_distributions")

## 7. Shift between MIMIC-IV and the external databases

This section supports the generalisation story: how different is each external database from the training
database? Three views, all on the `SHIFT_TASK` cohort:

* **Per variable:** standardised mean difference (SMD, |SMD| > 0.1 is the usual "meaningful imbalance" threshold),
  the Kolmogorov-Smirnov statistic between value distributions, and the difference in measurement coverage.
* **Statics and labels:** the same for age, height, weight, sex and the outcome.
* **Domain classifier:** LightGBM trained to tell the two databases apart from per-stay summaries. AUROC 0.5 means
  indistinguishable. It is fitted three times: on measurement counts only (how care is charted), on values only
  (physiology and units), and on both. Its top features show where the shift is concentrated.

In [ ]:
from scipy.stats import ks_2samp

OTHERS = [s for s in AVAILABLE.get(SHIFT_TASK, []) if s != REF_SOURCE]
RUN_SHIFT = REF_SOURCE in AVAILABLE.get(SHIFT_TASK, []) and bool(OTHERS)
if not RUN_SHIFT:
    print(f"Shift analysis skipped: needs {REF_SOURCE} plus another database in {SHIFT_TASK}.")


def smd(m1, s1, m2, s2):
    pooled = np.sqrt((s1 ** 2 + s2 ** 2) / 2)
    return (m2 - m1) / pooled if pooled > 0 else np.nan


SHIFT = {}
if RUN_SHIFT:
    R = STATS[(SHIFT_TASK, REF_SOURCE)]
    for tgt in OTHERS:
        T = STATS[(SHIFT_TASK, tgt)]
        rows = []
        for v in DYNAMIC_VARS:
            if v not in R.index or v not in T.index:
                continue
            ks = ks_2samp(SAMPLES[(SHIFT_TASK, REF_SOURCE)][v], SAMPLES[(SHIFT_TASK, tgt)][v])
            rows.append({"variable": v,
                         "SMD": smd(R.loc[v, "mean"], R.loc[v, "std"], T.loc[v, "mean"], T.loc[v, "std"]),
                         "KS": ks.statistic,
                         "median_ref": R.loc[v, "p50"], "median_tgt": T.loc[v, "p50"],
                         "coverage_ref_%": MISS[(SHIFT_TASK, REF_SOURCE)].loc[v, "rows_observed_%"],
                         "coverage_tgt_%": MISS[(SHIFT_TASK, tgt)].loc[v, "rows_observed_%"]})
        df = pd.DataFrame(rows).set_index("variable")
        df["coverage_diff_pp"] = df["coverage_tgt_%"] - df["coverage_ref_%"]
        df = df.reindex(df["SMD"].abs().sort_values(ascending=False).index)
        SHIFT[tgt] = save_table(df.round(3), f"shift_{REF_SOURCE}_vs_{tgt}")
        print(f"{NAMES[REF_SOURCE]} vs {NAMES[tgt]}: {int((df['SMD'].abs() > 0.1).sum())}/{len(df)} variables "
              f"with |SMD| > 0.1, median KS = {df['KS'].median():.3f}")
        display(df.round(2).head(15))

In [ ]:
NEG, POS = "#2a78d6", "#eb6834"   # diverging poles: lower / higher in the external database
if RUN_SHIFT:
    for tgt, df in SHIFT.items():
        d = df.sort_values("SMD")
        fig, axes = plt.subplots(1, 3, figsize=(WIDTH, 7.6), sharey=True)
        y = np.arange(len(d))
        axes[0].barh(y, d["SMD"], color=np.where(d["SMD"] > 0, POS, NEG), height=0.75)
        for x in (-0.1, 0.1):
            axes[0].axvline(x, color="#888", lw=0.8, ls=":")
        axes[0].axvline(0, color="#c3c2b7", lw=0.8)
        axes[0].set_title(f"SMD ({NAMES[tgt]} - {NAMES[REF_SOURCE]})")
        axes[1].barh(y, d["KS"], color="#6b6b6b", height=0.75)
        axes[1].set_title("KS statistic"); axes[1].set_xlim(0, 1)
        axes[2].barh(y, d["coverage_diff_pp"], color=np.where(d["coverage_diff_pp"] > 0, POS, NEG), height=0.75)
        axes[2].axvline(0, color="#c3c2b7", lw=0.8)
        axes[2].set_title("Coverage difference (pp)")
        axes[0].set_yticks(y, d.index)
        for ax in axes:
            ax.tick_params(axis="y", labelsize=7)
        fig.suptitle(f"Per-variable shift, {TASK_NAMES.get(SHIFT_TASK, SHIFT_TASK)} cohort "
                     f"(orange: higher in {NAMES[tgt]}, blue: lower)", y=1.0)
        fig.tight_layout()
        save(fig, f"shift_{REF_SOURCE}_vs_{tgt}_variables")

In [ ]:
if RUN_SHIFT:
    rows = []
    ref_sta = STA[(SHIFT_TASK, REF_SOURCE)]
    for tgt in OTHERS:
        tgt_sta = STA[(SHIFT_TASK, tgt)]
        for v in NUMERIC_STATICS:
            if v in ref_sta and v in tgt_sta:
                a = pd.to_numeric(ref_sta[v], errors="coerce").dropna()
                b = pd.to_numeric(tgt_sta[v], errors="coerce").dropna()
                if len(a) and len(b):
                    rows.append({"target": tgt, "variable": v, "ref": f"{a.median():.1f}", "tgt": f"{b.median():.1f}",
                                 "SMD": smd(a.mean(), a.std(), b.mean(), b.std()), "KS": ks_2samp(a, b).statistic})
        for task in TASKS:
            if (task, REF_SOURCE) in LABELS and (task, tgt) in LABELS:
                p1, p2 = LABELS[(task, REF_SOURCE)].mean(), LABELS[(task, tgt)].mean()
                h = 2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1))     # Cohen's h for proportions
                rows.append({"target": tgt, "variable": f"label_{task}", "ref": f"{100*p1:.1f}%",
                             "tgt": f"{100*p2:.1f}%", "SMD": h, "KS": np.nan})
        if "sex" in ref_sta and "sex" in tgt_sta:
            m1 = ref_sta["sex"].astype(str).str.lower().isin(["male", "m", "1"]).mean()
            m2 = tgt_sta["sex"].astype(str).str.lower().isin(["male", "m", "1"]).mean()
            rows.append({"target": tgt, "variable": "male", "ref": f"{100*m1:.1f}%", "tgt": f"{100*m2:.1f}%",
                         "SMD": 2 * np.arcsin(np.sqrt(m2)) - 2 * np.arcsin(np.sqrt(m1)), "KS": np.nan})
    static_shift = save_table(pd.DataFrame(rows), "shift_statics_and_labels", index=False)
    display(static_shift.round(3))

### Domain classifier

Features per stay (first 24 h of the `SHIFT_TASK` cohort): the mean of each variable, the number of measurements
of each variable, and the statics. 5-fold stratified cross-validation, at most `MAX_STAYS_CLASSIFIER` stays per
database so the classes stay balanced and the run stays fast.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def stay_features(src):
    f = STAY_AGG[src].copy()
    sta = STA[(SHIFT_TASK, src)].set_index("stay_id")
    for v in NUMERIC_STATICS:
        if v in sta:
            f[f"static_{v}"] = pd.to_numeric(sta[v], errors="coerce").reindex(f.index).astype("float32")
    if "sex" in sta:
        f["static_male"] = sta["sex"].astype(str).str.lower().isin(["male", "m", "1"]).reindex(f.index).astype("float32")
    if len(f) > MAX_STAYS_CLASSIFIER:
        f = f.sample(MAX_STAYS_CLASSIFIER, random_state=SEED)
    return f


def cv_auroc(X, y, n_splits=5):
    oof = np.zeros(len(y)); gains = np.zeros(X.shape[1])
    n_splits = int(min(n_splits, np.bincount(y).min()))
    for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(X, y):
        m = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, min_child_samples=20, n_jobs=N_THREADS, verbose=-1,
                           importance_type="gain", random_state=SEED)
        m.fit(X.iloc[tr], y[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:, 1]
        gains += m.feature_importances_
    return roc_auc_score(y, oof), pd.Series(gains / gains.sum(), index=X.columns)


DOMAIN = {}
if RUN_SHIFT:
    rows = []
    for tgt in OTHERS:
        fa, fb = stay_features(REF_SOURCE), stay_features(tgt)
        cols = sorted(set(fa.columns) & set(fb.columns))
        X = pd.concat([fa[cols], fb[cols]], ignore_index=True)
        y = np.r_[np.zeros(len(fa), int), np.ones(len(fb), int)]
        groups = {"counts only (charting process)": [c for c in cols if c.startswith("n_")],
                  "values only (physiology, units)": [c for c in cols if not c.startswith("n_")],
                  "all features": cols}
        for name, cs in groups.items():
            auc, imp = cv_auroc(X[cs], y)
            rows.append({"target": tgt, "features": name, "n_features": len(cs), "stays": len(y), "AUROC": auc})
            if name == "all features":
                DOMAIN[tgt] = imp.sort_values(ascending=False)
            print(f"{NAMES[REF_SOURCE]} vs {NAMES[tgt]} | {name:<32} AUROC = {auc:.3f}")
        del X, fa, fb
    domain_auc = save_table(pd.DataFrame(rows), "domain_classifier_auroc", index=False)
    display(domain_auc.round(3))
    for tgt, imp in DOMAIN.items():
        save_table(imp.rename("gain_share").to_frame(), f"domain_classifier_importance_{tgt}")

    fig, axes = plt.subplots(1, len(DOMAIN), figsize=(WIDTH * 0.55 * len(DOMAIN), 3.2), squeeze=False)
    for ax, (tgt, imp) in zip(axes[0], DOMAIN.items()):
        top = imp.head(15)[::-1]
        kind = ["#6250d6" if c.startswith("n_") else ("#1baf7a" if c.startswith("static_") else "#2a78d6")
                for c in top.index]
        ax.barh(np.arange(len(top)), 100 * top.values, color=kind, height=0.75)
        ax.set_yticks(np.arange(len(top)), top.index)
        auc = domain_auc[(domain_auc.target == tgt) & (domain_auc.features == "all features")]["AUROC"].iloc[0]
        ax.set_title(f"{NAMES[REF_SOURCE]} vs {NAMES[tgt]} (AUROC {auc:.2f})")
        ax.set_xlabel("Share of total gain (%)")
    from matplotlib.patches import Patch
    fig.tight_layout()
    fig.legend([Patch(color="#2a78d6"), Patch(color="#6250d6"), Patch(color="#1baf7a")],
               ["mean value", "number of measurements", "static"], frameon=False, loc="upper center",
               bbox_to_anchor=(0.5, 0.0), ncol=3)
    save(fig, "domain_classifier_importance")

## 8. Length of stay

Two views. **Raw ICU length of stay** from the PhysioNet tables (MIMIC-IV `icustays.los`, eICU
`patient.unitdischargeoffset`, HiRID/AUMCdb when available), restricted to the final cohort. **Hours of data in
the cohort** (`max(time) + 1` per stay in `dyn.parquet`), which is what the models actually see and which YAIB
caps per task.

We also look at the unit types (MIMIC-IV `first_careunit`, eICU `unittype`) since case mix differs between
databases.

In [ ]:
def _read_csv_cols(path, wanted, **kw):
    if not Path(path).exists():
        return None
    header = pd.read_csv(path, nrows=0, **kw).columns
    cols = [c for c in wanted if c in header]
    return pd.read_csv(path, usecols=cols, **kw) if cols else None


def raw_stay_extras(src) -> pd.DataFrame | None:
    """stay_id, los_days, unit (where the raw table has them)."""
    r = RAW / src
    if src == "miiv":
        df = _read_csv_cols(r / "icu" / "icustays.csv.gz", ["stay_id", "los", "first_careunit"])
        if df is None:
            return None
        df = df.rename(columns={"los": "los_days", "first_careunit": "unit"})
    elif src == "eicu":
        df = _read_csv_cols(r / "patient.csv.gz", ["patientunitstayid", "unitdischargeoffset", "unittype"])
        if df is None:
            return None
        df = df.rename(columns={"patientunitstayid": "stay_id", "unittype": "unit"})
        if "unitdischargeoffset" in df:
            df["los_days"] = df.pop("unitdischargeoffset") / 1440
    elif src == "hirid":
        return None          # HiRID general_table has no discharge time; LOS comes from the cohort hours below
    elif src == "aumc":
        df = _read_csv_cols(r / "admissions.csv", ["admissionid", "lengthofstay", "location"], encoding="latin-1")
        if df is None:
            return None
        df = df.rename(columns={"admissionid": "stay_id", "location": "unit"})
        if "lengthofstay" in df:
            df["los_days"] = df.pop("lengthofstay") / 24          # AUMCdb stores hours
    else:
        return None
    df["stay_id"] = df["stay_id"].astype(str)
    return df


EXTRAS = {}
for src in SOURCES:
    try:
        e = raw_stay_extras(src)
    except Exception as ex:
        print(f"[no raw extras] {src}: {type(ex).__name__}: {ex}"); e = None
    if e is not None and len(e.columns) > 1:
        EXTRAS[src] = e
        print(f"{src}: raw extras {list(e.columns)}")
    else:
        print(f"{src}: no LOS/unit columns in the raw tables here (synthetic data has none)")

rows = []
for task in TASKS:
    for src in AVAILABLE[task]:
        h = HOURS[(task, src)]
        r = {"task": task, "source": src, "cohort_hours_median": h.median(),
             "cohort_hours_IQR": f"{h.quantile(.25):.0f}-{h.quantile(.75):.0f}", "cohort_hours_max": h.max()}
        if src in EXTRAS and "los_days" in EXTRAS[src]:
            ids = set(pd.Series(KEEP[(task, src)]).astype(str))
            los = EXTRAS[src].loc[EXTRAS[src]["stay_id"].isin(ids), "los_days"].dropna()
            if len(los):
                r.update({"los_days_median": los.median(),
                          "los_days_IQR": f"{los.quantile(.25):.1f}-{los.quantile(.75):.1f}",
                          "los_>7d_%": 100 * (los > 7).mean()})
        rows.append(r)
los_table = save_table(pd.DataFrame(rows), "length_of_stay", index=False)
display(los_table.round(2))

In [ ]:
n_pan = len(TASKS) + (1 if EXTRAS else 0)
fig, axes = plt.subplots(1, n_pan, figsize=(WIDTH, 2.4), squeeze=False)
k = 0
if EXTRAS:
    ax = axes[0][k]; k += 1
    task = "aki" if "aki" in TASKS else TASKS[0]
    for src in EXTRAS:
        if "los_days" not in EXTRAS[src] or (task, src) not in KEEP:
            continue
        ids = set(pd.Series(KEEP[(task, src)]).astype(str))
        los = EXTRAS[src].loc[EXTRAS[src]["stay_id"].isin(ids), "los_days"].dropna()
        ax.hist(los.clip(0, 30), bins=np.linspace(0, 30, 61), density=True, histtype="step", lw=1.6,
                color=COLORS[src], label=NAMES[src])
    ax.set_xlabel("Days (clipped at 30)"); ax.set_yticks([])
    ax.set_title(f"Raw ICU LOS, {TASK_NAMES.get(task, task)} cohort")
for task in TASKS:
    ax = axes[0][k]; k += 1
    for src in AVAILABLE[task]:
        h = HOURS[(task, src)]
        ax.hist(h, bins=np.arange(0, h.max() + 2, max(1, int(h.max() // 40))), density=True, histtype="step",
                lw=1.6, color=COLORS[src], label=NAMES[src])
    ax.set_xlabel("Hours of data per stay"); ax.set_yticks([])
    ax.set_title(f"Hours of data, {TASK_NAMES.get(task, task)}")
fig.tight_layout()
fig_legend(fig, SOURCES)
save(fig, "length_of_stay")

In [ ]:
unit_srcs = [s for s in EXTRAS if "unit" in EXTRAS[s]]
if unit_srcs:
    task = SHIFT_TASK
    fig, axes = plt.subplots(1, len(unit_srcs), figsize=(WIDTH, 2.6), squeeze=False)
    rows = []
    for ax, src in zip(axes[0], unit_srcs):
        ids = set(pd.Series(KEEP.get((task, src), [])).astype(str))
        u = EXTRAS[src].loc[EXTRAS[src]["stay_id"].isin(ids), "unit"].fillna("unknown")
        share = (100 * u.value_counts(normalize=True)).head(8)[::-1]
        ax.barh(np.arange(len(share)), share.values, color=COLORS[src], height=0.7)
        ax.set_yticks(np.arange(len(share)), [s[:28] for s in share.index])
        ax.set_xlabel("% of stays"); ax.set_title(NAMES[src])
        rows += [{"source": src, "unit": k_, "share_%": v} for k_, v in share[::-1].items()]
    fig.suptitle(f"ICU type, {TASK_NAMES.get(task, task)} cohort (top 8)", y=1.0)
    fig.tight_layout()
    save(fig, "unit_types")
    display(save_table(pd.DataFrame(rows), "unit_types", index=False).round(1))

## 9. Summary table (Table 1 style)

Written as CSV and as a booktabs LaTeX table (`tables/summary.tex`) that can be `\input` in the thesis.

In [ ]:
def _get(df, task, src, col, fmt="{}"):
    r = df[(df.task == task) & (df.source == src)]
    if col not in r or r[col].isna().all():
        return "n/a"
    v = r[col].iloc[0]
    return fmt.format(v) if not isinstance(v, str) else v


summary_rows = []
for task in TASKS:
    for src in AVAILABLE[task]:
        summary_rows.append({
            "Task": TASK_NAMES.get(task, task), "Database": NAMES[src],
            "Stays": _get(attrition, task, src, "final", "{:,.0f}"),
            "Patients": _get(attrition, task, src, "final_patients", "{:,.0f}"),
            "Hospitals": _get(attrition, task, src, "hospitals", "{:,.0f}"),
            "Age, median [IQR]": _get(statics, task, src, "age_median[IQR]"),
            "Male (%)": _get(statics, task, src, "male_%", "{:.1f}"),
            "Positive (%)": _get(prevalence, task, src, "stay_prevalence_%", "{:.1f}"),
            "ICU LOS, days": _get(los_table, task, src, "los_days_median", "{:.1f}"),
            "Hours of data": _get(los_table, task, src, "cohort_hours_median", "{:.0f}"),
        })
summary = save_table(pd.DataFrame(summary_rows), "summary", index=False)
display(summary)


def to_booktabs(df: pd.DataFrame) -> str:
    esc = lambda s: str(s).replace("%", r"\%").replace("&", r"\&").replace("_", r"\_")
    lines = [r"\begin{tabular}{" + "l" * 2 + "r" * (df.shape[1] - 2) + "}", r"\toprule",
             " & ".join(esc(c) for c in df.columns) + r" \\", r"\midrule"]
    prev = None
    for _, r in df.iterrows():
        if prev is not None and r.iloc[0] != prev:
            lines.append(r"\midrule")
        lines.append(" & ".join(esc(v) for v in r.values) + r" \\")
        prev = r.iloc[0]
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines) + "\n"


(TAB / "summary.tex").write_text(to_booktabs(summary))

run_info = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "minutes": round((time.time() - T0) / 60, 1),
            "data": str(DATA), "tasks": TASKS, "sources": SOURCES, "first_stay_only": FIRST_STAY_ONLY,
            "figures": sorted(p.name for p in FIG.glob("*.pdf")), "tables": sorted(p.name for p in TAB.glob("*"))}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2))
try:
    import resource
    print(f"Peak memory: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.2f} GB")
except Exception:
    pass
print(json.dumps(run_info, indent=2))